<a id="inicio"></a>

# Airbnb Valencia — Business Intelligence Analysis

## Objective

Analyze the Valencia Airbnb market using a cloud-hosted PostgreSQL backend (Supabase) and a BI front-end (Preset / Apache Superset), delivering SQL-driven insights and interactive dashboards tailored to two stakeholder personas.

## Methodology

- **Data ingestion**: Load listings and reviews data from [Inside Airbnb](https://insideairbnb.com/get-the-data/) into Supabase-hosted PostgreSQL
- **Data modeling**: Clean and cast columns via `CREATE TABLE AS` to build analysis-ready `apartamentos_clean` and `reviews_clean` tables
- **Exploratory SQL analysis**: Answer business questions around pricing, geography, host concentration, and review dynamics
- **Deep-dive analytics**: Revenue evolution, feature-importance-style price drivers, event-window growth analysis (Fallas)
- **Dashboards in Preset**: Two dashboards — one for prospective investors, one for large property managers — exported to PDF

## Data source

Inside Airbnb — Valencia (not bundled with the repo due to redistribution terms). Download the `listings.csv.gz` and `reviews.csv.gz` files from [Inside Airbnb](https://insideairbnb.com/get-the-data/) and place them under `./data/`.

<div class="alert alert-block alert-info">

### ℹ️ Note on cell outputs

The cell outputs visible in this notebook were captured during the **original run** and are shown here as a visual reference for portfolio review. Some artifacts may still appear in Spanish because the outputs predate the English code translation.

**To re-run locally:**

1. Clone the repo and create a Python 3.9+ virtual environment:
   `python -m venv .venv && source .venv/bin/activate`
2. Install dependencies:
   `pip install psycopg2-binary pandas python-dotenv`
3. Download `listings.csv.gz` and `reviews.csv.gz` for Valencia from [Inside Airbnb](https://insideairbnb.com/get-the-data/) and extract them into `./data/`.
4. Set up a [Supabase](https://supabase.com/) project and load the CSVs into two tables (`apartamentos` and `reviews`). Capture the connection string from the Session Pooler settings.
5. Create a `.env` file in the project root with:
   ```
   SUPABASE_HOST=aws-0-eu-west-1.pooler.supabase.com
   SUPABASE_PORT=5432
   SUPABASE_DB=postgres
   SUPABASE_USER=your_username
   SUPABASE_PASSWORD=your_password
   ```
6. (Optional for dashboards) Create a [Preset](https://preset.io/) workspace, connect it to Supabase using the same credentials, and recreate the dashboards — PDFs from the original run are in `Resultados/`.

</div>

---

---

<a id="indice"></a>
## Contents

* [1. Use Case](#section1)
* [2. Dataset Generation](#section2)
* [3. Initial Analysis](#section3)
* [4. Deep Analysis](#section4)
* [5. Dashboards](#section5)
* [6. Business Scenarios](#section6)

---

<a id="section1"></a>
# 1. Use Case

In this project we analyze Airbnb listings in Valencia to better understand the short-term rental market. The goal is to explore the data and generate reports and dashboards that illuminate supply dynamics, pricing patterns, and demand trends.

Reference materials:
- [Inside Airbnb](https://insideairbnb.com/get-the-data/) — open Airbnb data for cities around the world
- [Data dictionary spreadsheet](https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit?gid=1322284596#gid=1322284596)

Step 1: download the Valencia `listings.csv.gz` and `reviews.csv.gz` from Inside Airbnb and place them in `./data/`.

---

<a id="section2"></a>
# 2. Dataset Generation

To host the data we use [Supabase](https://supabase.com/) — it provides a managed PostgreSQL backend with both programmatic access and built-in SQL editor. Setup:

1. Create a Supabase account
2. Start a new project
3. Grab the Session Pooler connection parameters

The code below connects to Supabase and creates the two initial tables for this analysis: one for listings and one for reviews.

In [ ]:
import os
from dotenv import load_dotenv
import psycopg2
import pandas as pd
load_dotenv()  # Load environment variables from .env



def get_connection():
    return psycopg2.connect(
        host=os.environ.get("SUPABASE_HOST", "aws-0-eu-west-1.pooler.supabase.com"),
        port=int(os.environ.get("SUPABASE_PORT", "5432")),
        dbname=os.environ.get("SUPABASE_DB", "postgres"),
        user=os.environ.get("SUPABASE_USER", "postgres"),
        password=os.environ["SUPABASE_PASSWORD"]
    )

In [ ]:
try:
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT 1;")
            result = cur.fetchone()
            print("Conexión exitosa. Resultado:", result)
except Exception as e:
    print("Error de conexión:", e)

In [ ]:
def create_table_from_df(sql):
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            conn.commit()

In [ ]:
def upload_table(df, sql, table_name, path):
    df.to_csv(path, index=False, quoting=1, na_rep="\\N")
    create_table_from_df(sql)
    with get_connection() as conn:
        with conn.cursor() as cur:
            with open(path, "r", encoding="utf-8") as f:
                cur.copy_expert(f'''
                    COPY {table_name} FROM STDIN WITH (FORMAT CSV, HEADER TRUE, QUOTE '"', NULL '\\N')
                ''', f)
            conn.commit()

In [ ]:
df_listings = pd.read_csv("./data/listings.csv")
apartamentos_create_table_sql = """
    CREATE TABLE IF NOT EXISTS apartamentos (
    id BIGINT,
    listing_url TEXT,
    scrape_id BIGINT,
    last_scraped TEXT,
    source TEXT,
    name TEXT,
    description TEXT,
    neighborhood_overview TEXT,
    picture_url TEXT,
    host_id BIGINT,
    host_url TEXT,
    host_name TEXT,
    host_since TEXT,
    host_location TEXT,
    host_about TEXT,
    host_response_time TEXT,
    host_response_rate TEXT,
    host_acceptance_rate TEXT,
    host_is_superhost TEXT,
    host_thumbnail_url TEXT,
    host_picture_url TEXT,
    host_neighbourhood TEXT,
    host_listings_count BIGINT,
    host_total_listings_count BIGINT,
    host_verifications TEXT,
    host_has_profile_pic TEXT,
    host_identity_verified TEXT,
    neighbourhood TEXT,
    neighbourhood_cleansed TEXT,
    neighbourhood_group_cleansed TEXT,
    latitude TEXT,
    longitude TEXT,
    property_type TEXT,
    room_type TEXT,
    accommodates BIGINT,
    bathrooms TEXT,
    bathrooms_text TEXT,
    bedrooms TEXT,
    beds TEXT,
    amenities TEXT,
    price TEXT,
    minimum_nights BIGINT,
    maximum_nights BIGINT,
    minimum_minimum_nights BIGINT,
    maximum_minimum_nights BIGINT,
    minimum_maximum_nights BIGINT,
    maximum_maximum_nights BIGINT,
    minimum_nights_avg_ntm TEXT,
    maximum_nights_avg_ntm TEXT,
    calendar_updated TEXT,
    has_availability TEXT,
    availability_30 BIGINT,
    availability_60 BIGINT,
    availability_90 BIGINT,
    availability_365 BIGINT,
    calendar_last_scraped TEXT,
    number_of_reviews BIGINT,
    number_of_reviews_ltm BIGINT,
    number_of_reviews_l30d BIGINT,
    availability_eoy BIGINT,
    number_of_reviews_ly BIGINT,
    estimated_occupancy_l365d BIGINT,
    estimated_revenue_l365d TEXT,
    first_review TEXT,
    last_review TEXT,
    review_scores_rating TEXT,
    review_scores_accuracy TEXT,
    review_scores_cleanliness TEXT,
    review_scores_checkin TEXT,
    review_scores_communication TEXT,
    review_scores_location TEXT,
    review_scores_value TEXT,
    license TEXT,
    instant_bookable TEXT,
    calculated_host_listings_count BIGINT,
    calculated_host_listings_count_entire_homes BIGINT,
    calculated_host_listings_count_private_rooms BIGINT,
    calculated_host_listings_count_shared_rooms BIGINT,
    reviews_per_month TEXT
    );
"""

In [ ]:
upload_table(df=df_listings, 
             sql=apartamentos_create_table_sql,
             table_name="apartamentos",
             path="./data/listings_airbnb.csv")

In [ ]:
df_reviews = pd.read_csv("./data/reviews.csv")
reviews_create_table_sql = """
    CREATE TABLE IF NOT EXISTS reviews (
    id BIGINT,
    listing_id BIGINT,
    date TEXT,
    reviewer_id BIGINT,
    reviewer_name TEXT,
    comments TEXT
    );
"""

In [11]:
df_reviews.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,48154,117554,2010-10-12,180238,Martha,Toni's place was perfect in so many ways. It ...
1,48154,145645,2010-11-28,204240,Mark,Awesome stay!! We'd recommend Toni's apartment...
2,48154,190572,2011-03-01,258565,Domenico,really nice house in a wonderfull position! yo...
3,48154,195081,2011-03-08,213496,Romina & Martín,"Apartamento muy agradable, al igual que su pro..."
4,48154,218435,2011-04-05,340330,Jenna,"Was a great apartment, easy access to the site..."


In [ ]:
upload_table(df=df_reviews, 
             sql=reviews_create_table_sql,
             table_name="reviews",
             path="./data/reviews_airbnb.csv")

### Quick sanity check on the loaded data

Query both tables to verify the data loaded correctly. Report:

- Row counts
- Column counts

Then, in a comment cell, briefly describe what each table contains.

In [ ]:
def check_table(table_name):
    with get_connection() as conn:
        with conn.cursor() as cur:
    
            cur.execute(f"SELECT COUNT(*) FROM {table_name};")
            row_count = cur.fetchone()[0]
            
   
            cur.execute(f"SELECT * FROM {table_name} LIMIT 1;")
            col_count = len(cur.description)
            
            print(f"Tabla '{table_name}': {row_count} filas, {col_count} columnas.")


check_table("apartamentos")
check_table("reviews")


### Analysis

The **`apartamentos`** table contains detailed information for each Airbnb listing in Valencia: location data, accommodation features (capacity, rooms, amenities), pricing, booking rules, and host details. Each row is one listing, which lets us analyze market supply and feature distributions.

The **`reviews`** table holds all guest reviews per listing — with the listing ID, date, reviewer identity, and comment text. This table supports reputation and satisfaction analysis and joins cleanly to `apartamentos` on the listing ID.

---

<a id="section3"></a>
# 3. Initial Analysis

For visualization we use [Preset.io](https://preset.io/) (a managed Apache Superset) — it handles database connections, dataset modeling, chart creation, and dashboard assembly. Setup:

1. Create a free Preset account
2. Create a new workspace named `CapstoneIX_<your_username>`
3. (In a classroom setting, share access with your instructor for traceability.)
4. Add a database connection of type `Other` that points to your Supabase instance via the Session Pooler connection string. Test the connection before saving.

### Cross-check in Preset SQL Lab

Run the same sanity queries in Preset's SQL Lab to confirm the data loaded correctly into the platform.

- How many rows and columns does each table have?

### Analysis

- **`apartamentos`**: 8,847 rows × 79 columns
- **`reviews`**: 416,389 rows × 6 columns

### Data type cleanup

Inspection reveals type issues in both tables — `price` is stored as text (with currency symbols), dates as strings, and so on. We create two new cleaned datasets (`apartamentos_clean`, `reviews_clean`) with properly-typed columns and only the fields needed for analysis.

Two key decisions per table:

- What's the correct dtype for each column?
- Which columns are actually useful?

### Analysis

**1. Cleaned apartments dataset**

```sql
CREATE TABLE public.apartamentos_clean AS
SELECT
    id::BIGINT AS id,
    name::VARCHAR(255) AS name,
    neighbourhood_cleansed::VARCHAR(100) AS neighbourhood_cleansed,
    CAST(NULLIF(NULLIF(NULLIF(latitude, '\N'), ''), 'nan') AS FLOAT) AS latitude,
    CAST(NULLIF(NULLIF(NULLIF(longitude, '\N'), ''), 'nan') AS FLOAT) AS longitude,
    property_type::VARCHAR(100) AS property_type,
    room_type::VARCHAR(100) AS room_type,
    CAST(CAST(NULLIF(NULLIF(NULLIF(accommodates::TEXT, '\N'), ''), 'nan') AS FLOAT) AS INT) AS accommodates,
    CAST(CAST(NULLIF(NULLIF(NULLIF(bedrooms, '\N'), ''), 'nan') AS FLOAT) AS INT) AS bedrooms,
    CAST(CAST(NULLIF(NULLIF(NULLIF(beds, '\N'), ''), 'nan') AS FLOAT) AS INT) AS beds,
    CASE
        WHEN price IN ('\N', '', 'nan') THEN NULL
        ELSE CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS NUMERIC(10,2))
    END AS price,
    CAST(CAST(NULLIF(NULLIF(NULLIF(minimum_nights::TEXT, '\N'), ''), 'nan') AS FLOAT) AS INT) AS minimum_nights,
    CAST(CAST(NULLIF(NULLIF(NULLIF(maximum_nights::TEXT, '\N'), ''), 'nan') AS FLOAT) AS INT) AS maximum_nights,
    CAST(CAST(NULLIF(NULLIF(NULLIF(number_of_reviews::TEXT, '\N'), ''), 'nan') AS FLOAT) AS INT) AS number_of_reviews,
    CAST(NULLIF(NULLIF(NULLIF(review_scores_rating, '\N'), ''), 'nan') AS FLOAT) AS review_scores_rating,
    host_id::BIGINT AS host_id,
    host_name::VARCHAR(255) AS host_name,
    CAST(NULLIF(NULLIF(NULLIF(host_since, '\N'), ''), 'nan') AS DATE) AS host_since,
    CASE
        WHEN host_is_superhost IN ('t', 'true', 'True', 'TRUE', 'yes', 'Yes', 'YES') THEN TRUE
        ELSE FALSE
    END AS host_is_superhost
FROM public.apartamentos;
```

**2. Cleaned reviews dataset**

```sql
CREATE TABLE public.reviews_clean AS
SELECT
    id::BIGINT AS id,
    listing_id::BIGINT AS listing_id,
    CAST(NULLIF(NULLIF(NULLIF(date, '\N'), ''), 'nan') AS DATE) AS date,
    reviewer_id::BIGINT AS reviewer_id,
    reviewer_name::VARCHAR(255) AS reviewer_name,
    comments::TEXT AS comments
FROM public.reviews;
```

**3. Column selection rationale — apartments**

Retained fields cover location, physical characteristics, pricing, reputation, and host identity — the core dimensions for market analysis: `id`, `name`, `neighbourhood_cleansed`, `latitude`, `longitude`, `property_type`, `room_type`, `accommodates`, `bedrooms`, `beds`, `price`, `minimum_nights`, `maximum_nights`, `number_of_reviews`, `review_scores_rating`, `host_id`, `host_name`, `host_since`, `host_is_superhost`.

Dropped fields include long-form descriptions, image URLs, host verification metadata, granular calendar fields, and auto-generated system metadata. These either had poor analytical value or duplicated signal already captured elsewhere.

**4. Column selection rationale — reviews**

All six columns kept: `id`, `listing_id`, `date`, `reviewer_id`, `reviewer_name`, `comments`. Each carries analytical signal — the comment text for NLP work later, the timestamp for trend analysis, the IDs for joins.

### Business questions

With cleaner datasets in place, we can start answering business questions via SQL. We'll tackle five:

- Where are the most expensive listings by property type?
- How has review volume evolved over time? Any anomalies?
- Which Valencia neighborhoods have the best and worst value-for-money?
- Where are the large-portfolio hosts concentrated?
- Which neighborhoods have grown the most in listings (and reviews) over the last 3 years?

For each SQL result, we also note which chart type would best communicate the insight.

### Note on data loading

`listing_id` and `id` got swapped during the initial Postgres load — likely a column-order mismatch at `CREATE TABLE` time. Fixed by permuting the columns in the final table.

### Analysis

**1. Most expensive listings by property type**

```sql
SELECT DISTINCT ON (property_type)
    property_type,
    neighbourhood_cleansed,
    price
FROM public.apartamentos_clean
WHERE price IS NOT NULL
ORDER BY property_type, price DESC;
```

A horizontal bar chart shows the top-priced listing per property type clearly — property types on the Y axis, max price on the X axis. Alternatively, a scatter map with points plotted by latitude/longitude surfaces the *geographic* distribution of these premium listings.

**2. Review volume over time**

```sql
SELECT
    DATE_TRUNC('month', date) AS mes,
    COUNT(*) AS num_reviews
FROM public.reviews_clean
WHERE date IS NOT NULL
GROUP BY mes
ORDER BY mes;
```

A time-series line chart is the natural fit — monthly review volume over the dataset's full range. Visible patterns: seasonality, growth trends, and shocks.

What jumps out:
- A dramatic drop in 2020 during COVID-19
- Rapid recovery and post-pandemic growth exceeding pre-pandemic levels
- Recent-month dips likely reflect incomplete review windows (reviews come in with a lag) rather than actual market contraction

Context and data-quality caveats matter when reading time-series like this.

**3. Best and worst value-for-money neighborhoods**

```sql
SELECT
    neighbourhood_cleansed,
    AVG(price) AS avg_price,
    AVG(review_scores_rating) AS avg_rating,
    (AVG(review_scores_rating) / NULLIF(AVG(price), 0)) AS value_for_money,
    COUNT(*) AS num_listings
FROM public.apartamentos_clean
WHERE price IS NOT NULL AND review_scores_rating IS NOT NULL
GROUP BY neighbourhood_cleansed
HAVING COUNT(*) > 10
ORDER BY value_for_money DESC;
```

Sorted horizontal bar chart by value-for-money ratio — lets readers compare quickly and see the tail in both directions.

**4. Large-portfolio host concentration by neighborhood**

```sql
WITH grandes_tenedores AS (
    SELECT host_id, COUNT(*) AS num_props
    FROM public.apartamentos_clean
    GROUP BY host_id
    HAVING COUNT(*) > 10  -- "Large" defined as 10+ listings (reasonable cutoff)
)
SELECT
    a.neighbourhood_cleansed,
    COUNT(DISTINCT a.host_id) AS num_grandes_tenedores,
    SUM(CASE WHEN g.host_id IS NOT NULL THEN 1 ELSE 0 END) AS num_propiedades_grandes_tenedores
FROM public.apartamentos_clean a
LEFT JOIN grandes_tenedores g ON a.host_id = g.host_id
GROUP BY a.neighbourhood_cleansed
ORDER BY num_propiedades_grandes_tenedores DESC;
```

Horizontal bar chart — lets us quickly see where institutional/professional hosts concentrate their inventory.

**5. Neighborhood growth (last 3 years)**

```sql
SELECT
    neighbourhood_cleansed,
    EXTRACT(YEAR FROM host_since) AS year,
    COUNT(*) AS num_new_props
FROM public.apartamentos_clean
WHERE host_since >= (CURRENT_DATE - INTERVAL '3 year')
GROUP BY neighbourhood_cleansed, year
ORDER BY num_new_props DESC;
```

Stacked bar chart (neighborhood × year) shows both overall growth and yearly pace.

---

<a id="section4"></a>
# 4. Deep Analysis

With preliminary patterns established, we go deeper. Questions:

- How has monthly revenue evolved by neighborhood?
- Which features drive listing price? (Intuition-led, then validated with SQL.)
- Which neighborhood grew the most *year-over-year* in reviews during Fallas, limited to listings with 4+ star ratings and 30+ reviews?

Each query again paired with a chart recommendation.

### Analysis

**1. Monthly revenue by neighborhood**

```sql
SELECT
    neighbourhood_cleansed,
    DATE_TRUNC('month', r.date) AS mes,
    SUM(a.price) AS ingresos_estimados
FROM public.reviews_clean r
JOIN public.apartamentos_clean a ON r.listing_id = a.id
WHERE r.date IS NOT NULL AND a.price IS NOT NULL
GROUP BY neighbourhood_cleansed, mes
ORDER BY mes, neighbourhood_cleansed;
```

Multi-line chart, one line per neighborhood. Captures both overall trend and relative market share over time.

**2. Price drivers**

```sql
-- Price vs. number of bedrooms
SELECT bedrooms, AVG(price) AS avg_price, COUNT(*) AS num_apartments
FROM public.apartamentos_clean
WHERE price IS NOT NULL AND bedrooms IS NOT NULL
GROUP BY bedrooms ORDER BY bedrooms;

-- Price vs. capacity (accommodates)
SELECT accommodates, AVG(price) AS avg_price, COUNT(*) AS num_apartments
FROM public.apartamentos_clean
WHERE price IS NOT NULL AND accommodates IS NOT NULL
GROUP BY accommodates ORDER BY accommodates;

-- Price vs. average rating
SELECT ROUND(review_scores_rating::numeric, 1) AS rating_rounded,
       AVG(price) AS avg_price, COUNT(*) AS num_apartments
FROM public.apartamentos_clean
WHERE price IS NOT NULL AND review_scores_rating IS NOT NULL
GROUP BY rating_rounded ORDER BY rating_rounded;

-- Price vs. Superhost status
SELECT host_is_superhost, AVG(price) AS avg_price, COUNT(*) AS num_apartments
FROM public.apartamentos_clean
WHERE price IS NOT NULL
GROUP BY host_is_superhost;
```

Line charts for the first three (showing how price responds to bedrooms / capacity / rating), a bar chart comparing Superhost vs. non-Superhost average prices.

Key findings:
- **Size and capacity push price up** — expected.
- **Surprisingly, Superhosts charge *less* on average**. Possible explanations: Superhosts prioritize occupancy and reviews over per-night premium, or they manage smaller / more budget-friendly properties. Either hypothesis would need deeper investigation to confirm.

**3. Fallas YoY review growth (rated 4+, 30+ reviews)**

```sql
WITH fallas_reviews AS (
    SELECT a.neighbourhood_cleansed AS barrio,
           EXTRACT(YEAR FROM r.date) AS anio,
           COUNT(*) AS num_reviews
    FROM public.reviews_clean r
    JOIN public.apartamentos_clean a ON r.listing_id = a.id
    WHERE EXTRACT(MONTH FROM r.date) = 3  -- March (Fallas)
      AND a.review_scores_rating >= 4
    GROUP BY a.neighbourhood_cleansed, EXTRACT(YEAR FROM r.date)
    HAVING COUNT(*) > 30
),
crecimiento AS (
    SELECT barrio, anio, num_reviews,
           LAG(num_reviews) OVER (PARTITION BY barrio ORDER BY anio) AS prev_year_reviews,
           num_reviews - LAG(num_reviews) OVER (PARTITION BY barrio ORDER BY anio) AS crecimiento
    FROM fallas_reviews
)
SELECT *
FROM crecimiento
WHERE crecimiento IS NOT NULL
ORDER BY crecimiento DESC
LIMIT 1;
```

Multi-line chart, one line per neighborhood, with Fallas-month review counts by year. Lets us see the growth leader clearly while also showing the broader trend across competitors.

---

<a id="section5"></a>
# 5. Dashboards

With the SQL analytics layer in place, the next step is dashboards. Two stakeholder personas, two dashboards.

### Stakeholder dashboards

**1. Prospective investor dashboard**

Audience: someone considering entering the Valencia short-term rental market. The dashboard needs to highlight where to invest, where growth is happening, and what profile of listings drive returns.

**Key KPIs**:
- Estimated annual revenue (price × review count)
- Occupancy proxy (review frequency)
- Neighborhood average rating
- Year-over-year demand growth (reviews or prices)

**Typical questions**:
- Which neighborhoods have the highest revenue per listing?
- Where are positive reviews growing fastest?
- Which property types are most profitable?
- How has supply in key areas shifted since 2020?

**Visuals**:
- Choropleth map of revenue per neighborhood
- Time series of review volume and average price
- Scatter: rating vs. review count per neighborhood

**2. Large property manager dashboard**

Audience: a host already running 10+ listings, looking to optimize their portfolio.

**Key KPIs**:
- Reviews per listing
- Rating per listing
- Month-over-month variation
- Distribution by room type and availability
- Alerts for drops in reviews or rating

**Typical questions**:
- Which listings have declining ratings?
- Which listings have stopped receiving reviews?
- Which listings get fewer than X reviews per month?
- Are there high-priced listings with poor reputation?

**Visuals**:
- Detailed table of listings with host filters
- Per-listing control panel (reviews, rating, price)
- Scatter: price vs. rating

Both dashboards were exported to PDF and live under `Resultados/` alongside this notebook.

---

<a id="section6"></a>
# 6. Business Scenarios

Beyond the cases above, here are three analytical scenarios where combining Airbnb data with external sources would yield richer insights — and how Preset/Superset would support them.

### Analysis

**Scenario 1 — Inflation impact on pricing and demand**

Cross Airbnb prices and review volume with macroeconomic indicators like CPI or unemployment rate. Datasets: `apartamentos_clean`, `reviews_clean`, plus external macro data. Metrics: price and review evolution vs. CPI — helps quantify how macro conditions shape the short-term rental market. Preset handles this well with temporal filters and multi-series charting.

**Scenario 2 — Airbnb vs. traditional hotels**

Compare Airbnb listings against hotel prices, ratings, and locations. Datasets: `apartamentos_clean`, `reviews_clean`, plus an external hotel dataset (e.g. Booking.com scrape or API). Visuals: price maps, rating comparisons, scatter plots — highlight value differences by area. Preset supports this with joins, interactive maps, and custom metrics.

**Scenario 3 — Emerging investment zones**

Cross Airbnb activity with urban planning and accessibility data (new construction permits, transit lines, etc.). Datasets: `apartamentos_clean`, `reviews_clean`, external urbanism and transport data. Metrics: review growth, low saturation, infrastructure improvements — identifies opportunity zones. Preset supports this via maps, neighborhood filters, and trend charts.

<div class="alert alert-block alert-success">

**Example extension: cultural events impact analysis**

Cross Airbnb reviews and pricing with local event calendars (Fallas, Mobile World Congress, San Fermín). Datasets: reviews, listings, plus an external events feed. Metrics: reviews per month, price variation, neighborhood booking rates before/during/after events — quantifies economic and operational impact of these events on short-term inventory.

Preset's temporal filters, geographic breakdowns, and period-vs-period comparisons handle this well.

</div>

### Data-quality and scaling considerations

**1. Technical barriers when scaling to multiple cities?**

Inconsistencies in neighborhood naming, date formats, currencies; uneven availability of reviews and prices across cities. Geographic granularity may differ, and some key fields may be missing in certain markets.

**2. Priority cities for expansion?**

Barcelona, Madrid, Seville, Málaga — high tourism volume, recurring events, dense Airbnb supply. These cities have enough data density to support meaningful comparisons.

**3. Structural differences between tourist and non-tourist cities?**

Tourist cities: higher listing concentration, more seasonal pricing, polarized ratings. Non-tourist cities: less rotation, more homogeneous supply.

**4. Demand evolution in each city's peak months?**

Summer and local-event peaks (Fallas, Holy Week). Seasonality varies by climate and international tourism appeal.

**5. Listing profiles that work best per city type?**

Central areas → studios and small apartments. Family-tourism or domestic-tourism cities → larger properties with multiple bedrooms.

**6. Common patterns vs. divergent behavior across equivalent neighborhoods?**

Central neighborhoods across cities tend to share high demand and prices but differ in rating and turnover. Peripheral neighborhoods vary dramatically in reputation and profitability.

**7. Segmenting guests by nationality or cultural profile via NLP on reviews?**

NLP can detect language, cultural references, preferences, and sentiment. Multilingual models can infer guest nationality and classify behavior or expectation patterns.

<div class="alert alert-block alert-success">

**Example extension: enriching with external signals**

Combining Airbnb data with weather, hotel pricing, or urban event calendars would contextualize demand and price changes beyond internal platform dynamics. Examples: explain localized price spikes via event calendars, explain review volume drops via weather data. Integrated external signals also power more robust predictive models and deliver a 360° view of the tourism market.

In Preset this is straightforward via dataset joins (or custom SQL), dynamic period filters, and comparative visualizations across internal vs. external dimensions.

</div>